In [1]:
import os
import sys
import json
import time
import socket
import shutil
import subprocess
from pathlib import Path
from pprint import pprint

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

def run_cmd(command, timeout=120):
    print(f"$ {command}")
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        timeout=timeout
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    return result

def check_port(host="127.0.0.1", port=11434):
    s = socket.socket()
    s.settimeout(2)
    try:
        s.connect((host, port))
        s.close()
        return True
    except Exception:
        return False

def section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

In [3]:
section("1. Vérification de la structure projet")

required_paths = [
    "app",
    "app/main.py",
    "app/core",
    "app/agents",
    "tests",
    "data/documents",
]

missing = [p for p in required_paths if not Path(p).exists()]
if missing:
    print("Éléments manquants :", missing)
else:
    print("Structure projet OK")


1. Vérification de la structure projet
Structure projet OK


In [4]:
section("2. Vérification des dépendances Python")

packages = [
    "streamlit",
    "langchain",
    "langchain_community",
    "langchain_huggingface",
    "faiss",
    "pypdf",
    "pytest"
]

results = {}
for pkg in packages:
    try:
        __import__(pkg)
        results[pkg] = "OK"
    except Exception as e:
        results[pkg] = f"KO: {e}"

pprint(results)


2. Vérification des dépendances Python
{'faiss': 'OK',
 'langchain': 'OK',
 'langchain_community': 'OK',
 'langchain_huggingface': 'OK',
 'pypdf': 'OK',
 'pytest': 'OK',
 'streamlit': 'OK'}


In [5]:
section("3. Vérification Ollama")

ollama_installed = shutil.which("ollama") is not None
print("Ollama installé :", ollama_installed)

if ollama_installed:
    run_cmd("ollama --version", timeout=30)
    print("Port 11434 ouvert :", check_port(port=11434))
else:
    print("Installe Ollama avant de continuer.")


3. Vérification Ollama
Ollama installé : True
$ ollama --version
ollama version is 0.32.9

Port 11434 ouvert : True


In [6]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest ⠋ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100

In [7]:
section("4. Vérification du modèle local")

if ollama_installed:
    result = run_cmd("ollama list", timeout=60)
    has_llama3 = "llama3" in result.stdout.lower()
    print("Modèle llama3 disponible :", has_llama3)
else:
    has_llama3 = False


4. Vérification du modèle local
$ ollama list
NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    

Modèle llama3 disponible : True


In [8]:
section("5. Vérification des documents")

pdfs = list(Path("data/documents").glob("*.pdf"))
print(f"Nombre de PDFs trouvés : {len(pdfs)}")
for pdf in pdfs[:10]:
    print("-", pdf.name)

if not pdfs:
    print("Ajoute au moins un PDF dans data/documents/")


5. Vérification des documents
Nombre de PDFs trouvés : 1
- GA_cours_Chap1_merged_unlocked.pdf


In [9]:
section("6. Chargement de la configuration")

from app.core.config import settings

config_dict = {
    "ollama_base_url": settings.ollama_base_url,
    "ollama_model": settings.ollama_model,
    "embedding_model": settings.embedding_model,
    "faiss_index_path": settings.faiss_index_path,
    "documents_path": settings.documents_path,
    "chunk_size": settings.chunk_size,
    "chunk_overlap": settings.chunk_overlap,
    "retriever_k": settings.retriever_k,
}
pprint(config_dict)


6. Chargement de la configuration
{'chunk_overlap': 64,
 'chunk_size': 512,
 'documents_path': 'data/documents',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'faiss_index_path': 'storage/faiss_index',
 'ollama_base_url': 'http://localhost:11434',
 'ollama_model': 'llama3.2',
 'retriever_k': 4}


In [10]:
section("7. Test document loader + splitter")

from app.core.document_loader import load_documents, split_documents

docs = load_documents()
print("Nombre de pages chargées :", len(docs))

chunks = split_documents(docs) if docs else []
print("Nombre de chunks :", len(chunks))

if chunks:
    print("\nExemple de chunk :")
    print(chunks[0].page_content[:500])
    print("\nMetadata :")
    pprint(chunks[0].metadata)


7. Test document loader + splitter
[2026-08-12 16:00:44] INFO [app.core.document_loader] Loading: GA_cours_Chap1_merged_unlocked.pdf
[2026-08-12 16:00:46] INFO [app.core.document_loader] Loaded 45 pages from 1 PDFs
Nombre de pages chargées : 45
[2026-08-12 16:00:46] INFO [app.core.document_loader] Split into 245 chunks
Nombre de chunks : 245

Exemple de chunk :
Chapitre 1
Graphes non-orientés
Table des matières
1 Définitions 2
2 Connectivité d’un graphe 3
3 Arbres 3
3.1 Définition(s) . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3
3.2 Comment faire pousser un arbre . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4
3.3 Nombre d’arbres . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4

Metadata :
{'page': 0, 'source': 'data\\documents\\GA_cours_Chap1_merged_unlocked.pdf'}


In [11]:
section("8. Test embeddings + FAISS")

from pathlib import Path
from app.core.vector_store import build_vector_store, load_vector_store

store = None
index_path = Path(settings.faiss_index_path)

if chunks:
    store = build_vector_store(chunks)
    print("FAISS index créé avec succès.")
    print("Index path existe :", index_path.exists())
else:
    print("Impossible de construire FAISS : aucun chunk.")


8. Test embeddings + FAISS
[2026-08-12 16:00:46] INFO [app.core.vector_store] Building FAISS index...
[2026-08-12 16:00:46] INFO [app.core.embeddings] Loading embeddings model: sentence-transformers/all-MiniLM-L6-v2


c:\Users\amind\AppData\Local\Programs\Python\Python312\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[2026-08-12 16:01:23] INFO [app.core.vector_store] FAISS index saved to storage/faiss_index
FAISS index créé avec succès.
Index path existe : True


In [12]:
section("9. Test retriever")

from app.core.retriever import get_retriever

if store is None and index_path.exists():
    store = load_vector_store()

if store is not None:
    retriever = get_retriever(store)
    test_query = "Donne-moi les principaux sujets abordés dans les documents."
    retrieved_docs = retriever.invoke(test_query)
    print(f"Nombre de documents récupérés : {len(retrieved_docs)}")
    for i, d in enumerate(retrieved_docs[:3], start=1):
        print(f"\n--- Document récupéré #{i} ---")
        print(d.page_content[:400])
        pprint(d.metadata)
else:
    print("Retriever non testé : store indisponible.")


9. Test retriever
Nombre de documents récupérés : 4

--- Document récupéré #1 ---
2.
Comme on va le montrer ci-après, c’est le pire qui puisse arriver.
10
{'page': 9, 'source': 'data\\documents\\GA_cours_Chap1_merged_unlocked.pdf'}

--- Document récupéré #2 ---
une priorité entre les noeuds (si on affecté un label aux noeuds au préalable), ou aléatoire. On peut par exemple
choisir le noeud v:
—Uniformément au hasard dansVà chaque étape. On parle alors d’algorithme glouton;
— Avec une certaine mesure de probabilité arbitraire parmi les noeuds labellisés;
8
{'page': 7, 'source': 'data\\documents\\GA_cours_Chap1_merged_unlocked.pdf'}

--- Document récupéré #3 ---
une carte géographique, représentons par un noeud d’un graphe, chaque pays, et par une arête chaque frontière
entre 2 pays. Le graphe obtenu est nécessairement planaire, et la carte correspondante est appelée carte duale
du graphe. On a le résultat culte suivant, dont on ne saurait résumer la preuve (qui s’étend sur des dizaines

In [14]:
section("10. Test agent LangChain")

from app.core.memory import get_memory
from app.agents.agent_executor import create_agent

agent = None

if store is not None:
    memory = get_memory()
    retriever = get_retriever(store)
    agent = create_agent(retriever, memory)
    print("Agent créé avec succès.")
else:
    print("Agent non créé : store indisponible.")


10. Test agent LangChain


c:\Users\amind\AppData\Local\Programs\Python\Python312\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


Agent créé avec succès.


In [16]:
from app.agents.summarizer_tool import create_summarizer_tool

summarizer_tool = create_summarizer_tool(retriever)

result = summarizer_tool.invoke(
    "Résume le contenu principal des documents"
)

print(result)

[2026-08-12 16:03:00] INFO [app.agents.summarizer_tool] Recherche de documents pour résumé : Résume le contenu principal des documents
Synthèse générale :

Le contexte fourni présente un document qui semble être lié à la théorie des graphes et plus spécifiquement à l'algorithme de construction d'un couplage maximal dans un graphe. Le document contient plusieurs parties qui traitent de différentes étapes de ce processus.

Points clés :

1. L'algorithme commence par effacer les arêtes de Gune jusqu'à ce que le sous-graphe obtenu ne soit plus connexe, ce qui signifie qu'il n'y a plus d'arêtes entre les noeuds.
2. Le document mentionne une caractérisation (ii) de la Proposition 2, mais il n'est pas clair quel est l'objet de cette proposition ou comment elle est liée à l'algorithme.
3. L'algorithme utilise un processus de boucle pour effacer les demi-arêtes disponibles, en commençant par le paquet 1 et en augmentant progressivement le nombre de paquets non vides.
4. Le document mentionne qu

In [15]:
section("11. Questions de validation agent")

test_questions = [
    "Résume le contenu principal des documents.",
    "Quel est le sujet principal du corpus ?",
    "Calcule 25 * 14 + 8",
]

if agent is not None:
    for q in test_questions:
        print(f"\nQUESTION: {q}")
        try:
            result = agent.invoke({"input": q})
            output = result.get("output", result)
            print("RÉPONSE:")
            print(output)
        except Exception as e:
            print("Erreur :", e)
else:
    print("Agent indisponible.")


11. Questions de validation agent

QUESTION: Résume le contenu principal des documents.


> Entering new AgentExecutor chain...


Thought: Do I need to use a tool? Yes
Action: Summarizer
Action Input: le contenu principal des documentsJe suis désolé, mais il semble que vous n'avez pas fourni de texte pour résumer. Voudriez-vous me donner le texte à résumer ou donner un sujet précis sur lequel je devrais résumer ? Je serai ravi de l'aider !Final Answer: Pouvez-vous s'il vous plaît fournir le contenu principal des documents ou spécifier un sujet précis pour lesquels je devrais résumer ? Je serai ravi de vous aider !

> Finished chain.
RÉPONSE:
Pouvez-vous s'il vous plaît fournir le contenu principal des documents ou spécifier un sujet précis pour lesquels je devrais résumer ? Je serai ravi de vous aider !

QUESTION: Quel est le sujet principal du corpus ?


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: DocumentSearch
Action Input: Quel est le sujet principal du corpus ?

C:\Users\amind\Documents\3A\ML\multi-agent-rag-assistant\app\agents\rag_tool.py:19: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  func=lambda q: chain({"question": q}),


{'question': 'Quel est le sujet principal du corpus ?', 'answer': "I don't know the answer to your question about which state/country's law governs the interpretation of a contract.\n\n", 'sources': '', 'source_documents': [Document(metadata={'source': 'data\\documents\\GA_cours_Chap1_merged_unlocked.pdf', 'page': 7}, page_content='En d’autre termes, une famille indépendante Iest maximale si tous les noeuds qui n’y appartiennent pas\nfont partie de son voisinage, i.e., E(I) =V \\ I. Trouver une famille indépendante maximale dans un graphe est\nun problème courant en pratique : en télécommunications sans fil, on peut représenter par des noeuds les agents\nqui émettent/reçoivent sur le réseau. Les arêtes représentent la trop grande proximité géographique entre agents,'), Document(metadata={'source': 'data\\documents\\GA_cours_Chap1_merged_unlocked.pdf', 'page': 9}, page_content='2.\nComme on va le montrer ci-après, c’est le pire qui puisse arriver.\n10'), Document(metadata={'source': 'da